# SXF — Isolated Vision Verification (Google Colab)

**Follow-up to the completed SXF Tesla T4 benchmark.** The original Qwen3 8B and Qwen3-VL 4B files remain unchanged. This notebook checks the outstanding issues: whether the **vision model is loaded on GPU without the text model resident**, and whether its **actual image answer** matches the test image. It repeats only the **vision** measurement, not both models.

This runs on a **Google Colab NVIDIA GPU, not your MacBook**. In **Runtime → Change runtime type → T4 GPU**, choose an available GPU. Then choose **Runtime → Run all**. Free GPU allocation may be unavailable; **do not pay for GPU unless you choose to**. The Ollama model is pulled into the temporary cloud VM only.

**Evidence and limitations:** The output retains the unedited model response for human inspection, local Ollama /api/ps residency metadata, runtime version and GPU before/after memory snapshots. This is stronger evidence than the first capture but does **not** prove physical origin, GPU-exclusive allocation, comprehensive visual accuracy, or GPU residency of every layer. GPU-memory deltas are observed device totals, not model allocations. Neither this notebook nor any script approves or publishes results to sxf.si.

In [ ]:
import pathlib, subprocess, sys, shutil, importlib, urllib.request, json, time
ROOT=pathlib.Path("/content/sxf-ai")
OUTPUT=pathlib.Path("/content/sxf-vision-verification")
OUTPUT.mkdir(parents=True,exist_ok=True)
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Colab has no NVIDIA GPU. Set Runtime → Change runtime type → T4 GPU.")
probe=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],text=True,capture_output=True,check=True)
gpu_rows=[x.strip() for x in probe.stdout.splitlines() if x.strip()]
if len(gpu_rows)!=1:
    raise RuntimeError("Only one NVIDIA GPU is supported for this evidence procedure.")
print("Cloud GPU:",gpu_rows[0])
if not (ROOT/".git").exists():
    subprocess.run(["git","clone","--depth","1","--branch","main","https://github.com/rab2323s-creator/sxf-ai.git",str(ROOT)],check=True)
else:
    subprocess.run(["git","-C",str(ROOT),"fetch","origin","main"],check=True)
    subprocess.run(["git","-C",str(ROOT),"merge","--ff-only","FETCH_HEAD"],check=True)
sys.path.insert(0,str(ROOT/"scripts"))
import ensure_colab_ollama
importlib.reload(ensure_colab_ollama)
OLLAMA=ensure_colab_ollama.ensure_ollama_ready(OUTPUT)
print("Ready to collect isolated Vision evidence.")

In [ ]:
import urllib.request, json, time, subprocess
API="http://127.0.0.1:11434"
def api(path,payload=None):
    data=None if payload is None else json.dumps(payload).encode("utf-8")
    req=urllib.request.Request(API+path,data=data,headers={"Content-Type":"application/json"})
    with urllib.request.urlopen(req,timeout=600) as response: return json.load(response)
def gpu_memory():
    out=subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.used,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True,timeout=15)
    rows=[row for row in out.stdout.splitlines() if row.strip()]
    if len(rows)!=1:raise RuntimeError("GPU telemetry is not single-device.")
    parts=[x.strip() for x in rows[0].split(",")]
    if len(parts)!=4:raise RuntimeError("Unexpected NVIDIA telemetry fields.")
    return {"name":parts[0],"totalMiB":int(parts[1]),"usedMiB":int(parts[2]),"driverVersion":parts[3]}

# Clear all resident model sessions, including the old qwen3:8b benchmark.
residents=api("/api/ps").get("models",[])
for model in residents:
    name=model.get("name") or model.get("model")
    if not name:raise RuntimeError("Unknown resident model; cannot unload safely.")
    print("Unloading resident model:",name,flush=True)
    api("/api/generate",{"model":name,"keep_alive":0,"stream":False})
for attempt in range(45):
    remaining=api("/api/ps").get("models",[])
    if not remaining:break
    time.sleep(1)
else:
    raise RuntimeError("Models are still resident; refusing a contaminated baseline.")
idle_gpu=gpu_memory()
print("No Ollama models resident. Baseline used GPU memory:",idle_gpu["usedMiB"],"MiB")

In [ ]:
import struct,zlib,hashlib,base64,datetime,subprocess
w=h=128
raw=b"".join(b"\x00"+b"".join((bytes((238,42,42)) if 24<=x<104 and 24<=y<104 else bytes((36,77,208))) for x in range(w)) for y in range(h))
def png_chunk(tag,data):
    return struct.pack("!I",len(data))+tag+data+struct.pack("!I",zlib.crc32(tag+data)&0xffffffff)
image=b"\x89PNG\r\n\x1a\n"+png_chunk(b"IHDR",struct.pack("!2I5B",w,h,8,2,0,0,0))+png_chunk(b"IDAT",zlib.compress(raw))+png_chunk(b"IEND",b"")
fixture=OUTPUT/"vision-square.png"
fixture.write_bytes(image)
image_hash=hashlib.sha256(image).hexdigest()
if api("/api/ps").get("models"):
    raise RuntimeError("Another Ollama model reloaded before Vision validation. Stop here.")
subprocess.run([OLLAMA,"pull","qwen3-vl:4b"],check=True)
installed=[m for m in api("/api/tags").get("models",[]) if m.get("name")=="qwen3-vl:4b" or m.get("model")=="qwen3-vl:4b"]
if len(installed)!=1:
    raise RuntimeError("Cannot unambiguously identify exact installed qwen3-vl:4b tag.")
identity=installed[0]
start=time.monotonic()
response=api("/api/generate",{
    "model":"qwen3-vl:4b",
    "prompt":"Describe the main visible objects, shapes and colors in this image.",
    "images":[base64.b64encode(image).decode("ascii")],
    "think":False,"stream":False,"keep_alive":"15m",
    "options":{"num_ctx":4096,"temperature":0,"num_predict":192}
})
duration=time.monotonic()-start
answer=response.get("response")
if response.get("done") is not True or not isinstance(answer,str) or not answer.strip():
    raise RuntimeError("Vision inference returned no usable answer; do not certify vision capability.")
resident=api("/api/ps").get("models",[])
match=[m for m in resident if m.get("name")=="qwen3-vl:4b" or m.get("model")=="qwen3-vl:4b"]
if len(match)!=1:
    raise RuntimeError("No unambiguous vision model found in Ollama /api/ps after inference.")
after_gpu=gpu_memory()
print("Actual Vision answer (needs manual review):",answer)
print("Observed whole-GPU memory before/after:",idle_gpu["usedMiB"],"/",after_gpu["usedMiB"],"MiB")
print("Ollama reported model size / GPU-resident bytes:",match[0].get("size"),"/",match[0].get("size_vram"))
report={
    "schemaVersion":1,"kind":"vision-runtime-verification",
    "reviewStatus":"manual-review-required","measuredAt":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "modelId":"ollama:qwen3-vl:4b","modelTag":"qwen3-vl:4b",
    "modelDigest":identity.get("digest"),"modelSizeBytes":identity.get("size"),
    "reportedQuantization":identity.get("details",{}).get("quantization_level"),
    "runtime":"Ollama","runtimeVersion":api("/api/version").get("version"),
    "imageSha256":image_hash,"imageBytes":len(image),
    "imageExpectedDescription":"One red square centered over a blue background.",
    "visionAnswer":answer,"answerAccuracy":"not-independently-reviewed",
    "inferenceWallSeconds":round(duration,3),
    "requestedContextTokens":4096,"requestedThink":False,
    "gpuBaselineAfterModelUnload":idle_gpu,"gpuAfterVisionLoad":after_gpu,
    "ollamaPsAfterVisionInference":match[0],
    "memoryEvidenceNotes":"Unloaded other Ollama models first; before/after values are total GPU memory. Ollama /api/ps size_vram is runtime-reported and does not independently prove all layers are on GPU."
}
output=OUTPUT/"vision-runtime-verification.json"
if output.exists():raise FileExistsError("Do not overwrite existing verification. Start a fresh Colab session for a new capture.")
output.write_text(json.dumps(report,indent=2)+"\n")
print("Saved Vision verification:",output)

In [ ]:
import subprocess,sys,json
measure=OUTPUT/"qwen3-vl-4b-isolated-4096.json"
if not measure.exists():
    run=subprocess.run([sys.executable,"-u","scripts/measure_local_ollama.py",
          "--model-id","ollama:qwen3-vl:4b","--context","4096","--repeats","3",
          "--image",str(fixture),"--output",str(measure)],cwd=ROOT,capture_output=True,text=True)
    if run.stdout:print(run.stdout[-16000:],flush=True)
    if run.stderr:print(run.stderr[-16000:],flush=True)
    if run.returncode:raise RuntimeError("Isolated Vision measurement failed; see diagnostics above (status "+str(run.returncode)+").")
trial=json.loads(measure.read_text())
if trial["kind"]!="local-measurement-candidate" or trial["publicationStatus"]!="review-required" or trial["repetitions"]<3 or trial["inputImageSha256"]!=image_hash:
    raise RuntimeError("Measured candidate does not match fixture or review gate.")
print("Isolated Vision speed trial captured; no results were published.")

In [ ]:
import datetime,hashlib,json,zipfile,pathlib
from google.colab import files
names=["vision-runtime-verification.json","qwen3-vl-4b-isolated-4096.json","vision-square.png"]
for name in names:
    if not (OUTPUT/name).is_file():
        raise RuntimeError("Missing required real evidence: "+name)
manifest={"kind":"sxf-private-evidence-for-review","reviewStatus":"manual-review-required",
          "createdAtUtc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "source":"SXF isolated Vision follow-up in user-controlled Colab GPU session",
          "files":[{"name":n,"sha256":hashlib.sha256((OUTPUT/n).read_bytes()).hexdigest()} for n in names]}
(OUTPUT/"manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
archive=pathlib.Path("/content/sxf-isolated-vision-evidence.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for name in names+["manifest.json"]:z.write(OUTPUT/name,name)
print("Ready for private evidence review (NOT published):",archive)
files.download(str(archive))

### After the ZIP downloads

Send **sxf-isolated-vision-evidence.zip** back in your original SXF ChatGPT conversation. Keep your previous **sxf-cloud-gpu-evidence.zip** as separate, unmodified first-run evidence. The reviewer will check the image answer and report GPU/model residency conservatively before considering a public, source-linked benchmark entry. **Nothing is automatically approved or published.**